# 07. Pipeline de Inferencia y Priorización para Analista — IBM TabFormer
**Unidad 1:** Detección de Fraude en Transacciones con Tarjeta de Crédito  
**Dataset:** Lote de Inferencia Desacoplado (`sample_inference.csv`)  
**Entorno:** Python 3.12 | scikit-learn | Polars

---

### Objetivos del Notebook:
1. Cargar el preprocesador serializado (`preprocessor.joblib`) y el modelo baseline (`baseline_logistic_regression.joblib`).
2. Recibir un lote de transacciones nuevas desde `data/processed/sample_inference.csv`.
3. Validar el contrato de datos (confirmando que **no se requiere la columna `Is Fraud?`**).
4. **Regla de Negocio Fuera de Cohorte (`OUT_OF_COHORT_REVIEW`):** Identificar transacciones con `Amount <= 0` (reembolsos, contracargos o ajustes) y separarlas para auditoría manual por reglas operativas.
5. Aplicar las transformaciones aprendidas sobre las transacciones de compra positiva (`Amount > 0`).
6. Calcular el **Risk Score (0 a 100)** y aclarar formalmente que es una puntuación de ordenamiento de riesgo (no una probabilidad calibrada).
7. Asignar niveles de prioridad para la revisión del analista humano (`ALTA`, `MEDIA`, `BAJA`, `OUT_OF_COHORT_REVIEW`).
8. Preservar `transaction_id` para asegurar la trazabilidad operativa y exportar a `data/predictions/inference_results.csv`.


In [1]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR
while not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from notebooks._comun.config import (
    INFERENCE_SAMPLE_FILE,
    PREPROCESSOR_ARTIFACT,
    LOGISTIC_MODEL_ARTIFACT,
    INFERENCE_RESULTS_FILE,
    SELECTED_FEATURES,
)
from notebooks._comun.data_utils import load_artifact
print(f"Raíz del proyecto: {PROJECT_ROOT}")


Raíz del proyecto: /home/glender/ProyectosUNi/iaupeu


In [2]:
import polars as pl
import numpy as np
import pandas as pd

# Carga de artefactos
preprocessor = load_artifact(PREPROCESSOR_ARTIFACT)
model = load_artifact(LOGISTIC_MODEL_ARTIFACT)

print("Preprocesador y Modelo Baseline cargados exitosamente.")


Preprocesador y Modelo Baseline cargados exitosamente.


## 1. Carga del Lote de Transacciones Nuevas
Cargamos el archivo CSV de transacciones a evaluar (`sample_inference.csv`).


In [3]:
# Lectura del lote
df_new = pl.read_csv(INFERENCE_SAMPLE_FILE)
print(f"Transacciones recibidas para inferencia: {df_new.shape[0]:,} filas x {df_new.shape[1]} columnas")

# Verificación de que la etiqueta no está presente ni es requerida
assert "Is Fraud?" not in df_new.columns and "is_fraud" not in df_new.columns, \
    "Error: El lote de inferencia no debe contener la etiqueta de fraude."
print("Validación exitosa: El flujo opera sin etiqueta de fraude.")


Transacciones recibidas para inferencia: 1,000 filas x 20 columnas
Validación exitosa: El flujo opera sin etiqueta de fraude.


## 2. Validación de Contrato de Datos y Regla `OUT_OF_COHORT_REVIEW`
- **Regla Operativa:** Cualquier transacción con `Amount <= 0` (reembolso o transacción de control cero) no pertenece a la cohorte de compras entrenada.
- **Acción:** Se cataloga automáticamente como `OUT_OF_COHORT_REVIEW` y se deriva a la cola de auditoría de devoluciones.


In [4]:
missing_features = [col for col in SELECTED_FEATURES if col not in df_new.columns]
if missing_features:
    raise ValueError(f"Faltan características obligatorias en la entrada: {missing_features}")

# Separar cohorte válida de compra (Amount > 0) y excepciones (Amount <= 0)
mask_out_of_cohort = (df_new["Amount"] <= 0).to_numpy()
n_out = int(np.sum(mask_out_of_cohort))
n_valid = int(len(df_new) - n_out)

print(f"Transacciones válidas de compra (Amount > 0): {n_valid:,}")
print(f"Transacciones fuera de cohorte (Amount <= 0):     {n_out:,}")


Transacciones válidas de compra (Amount > 0): 995
Transacciones fuera de cohorte (Amount <= 0):     5


## 3. Transformación de Características y Scoring
Aplicamos `preprocessor.transform()` sobre las transacciones de compra positiva.

> **Nota Metodológica sobre `risk_score`:**
> La puntuación derivada de la Regresión Logística con `class_weight='balanced'` representa un **score de ordenamiento de riesgo relativo (*risk_score*)**, no una probabilidad bayesiana exactamente calibrada a la prevalencia marginal de 0.12%.
> Por ende, los umbrales de corte (70 y 40) en esta etapa baseline son **referenciales y provisionales**, destinados a priorizar la cola de alertas para el analista humano.


In [5]:
# Arrays para resultados completos
probabilities = np.zeros(len(df_new), dtype=float)
predictions = np.zeros(len(df_new), dtype=int)
risk_scores = np.zeros(len(df_new), dtype=float)
priority_levels = [""] * len(df_new)
acciones_recomendadas = [""] * len(df_new)

# Inferencia sobre transacciones válidas
idx_valid = np.where(~mask_out_of_cohort)[0]
if len(idx_valid) > 0:
    df_valid = df_new.filter(pl.col("Amount") > 0)
    X_valid_df = df_valid.select(SELECTED_FEATURES).to_pandas()
    X_trans = preprocessor.transform(X_valid_df)

    probs_valid = model.predict_proba(X_trans)[:, 1]
    preds_valid = model.predict(X_trans)
    scores_valid = np.round(probs_valid * 100, 2)

    for i, idx in enumerate(idx_valid):
        sc = scores_valid[i]
        probabilities[idx] = round(probs_valid[i], 4)
        predictions[idx] = int(preds_valid[i])
        risk_scores[idx] = sc

        if sc >= 70.0:
            priority_levels[idx] = "ALTA PRIORIDAD"
            acciones_recomendadas[idx] = "Revisión inmediata / retención preventiva preventiva y contacto telefónico"
        elif sc >= 40.0:
            priority_levels[idx] = "MEDIA PRIORIDAD"
            acciones_recomendadas[idx] = "Monitoreo reforzado / solicitar desafío de autenticación 2FA"
        else:
            priority_levels[idx] = "BAJA PRIORIDAD"
            acciones_recomendadas[idx] = "Aprobación automática estándar"

# Asignación a casos fuera de cohorte
idx_out = np.where(mask_out_of_cohort)[0]
for idx in idx_out:
    probabilities[idx] = 0.0
    predictions[idx] = -1
    risk_scores[idx] = 0.0
    priority_levels[idx] = "OUT_OF_COHORT_REVIEW"
    acciones_recomendadas[idx] = "Auditoría manual de reembolso / contracargo por reglas operativas"

print(f"Scoring completado sobre {len(df_new)} transacciones.")


Scoring completado sobre 1000 transacciones.


## 4. Priorización para Revisión Humana
El analista humano dispone de una cola de trabajo estructurada por nivel de prioridad.


In [6]:
# Construir DataFrame de resultados con trazabilidad
df_results = df_new.with_columns([
    pl.Series("fraud_probability", probabilities),
    pl.Series("risk_score", risk_scores),
    pl.Series("prediction", predictions),
    pl.Series("priority_level", priority_levels),
    pl.Series("accion_recomendada", acciones_recomendadas),
])

print("Resumen de Transacciones por Nivel de Prioridad:")
print(df_results.group_by("priority_level").len().sort("len", descending=True))

print("\nTransacciones clasificadas como OUT_OF_COHORT_REVIEW:")
print(
    df_results.filter(pl.col("priority_level") == "OUT_OF_COHORT_REVIEW")
    .select(["transaction_id", "Amount", "Use Chip", "priority_level", "accion_recomendada"])
)

print("\nTop 5 transacciones de mayor riesgo:")
top_risky = (
    df_results.filter(pl.col("priority_level") != "OUT_OF_COHORT_REVIEW")
    .sort("risk_score", descending=True)
    .select(["transaction_id", "Amount", "Use Chip", "mcc_category", "hour", "risk_score", "priority_level"])
    .head(5)
)
print(top_risky)


Resumen de Transacciones por Nivel de Prioridad:
shape: (4, 2)
┌──────────────────────┬─────┐
│ priority_level       ┆ len │
│ ---                  ┆ --- │
│ str                  ┆ u32 │
╞══════════════════════╪═════╡
│ BAJA PRIORIDAD       ┆ 597 │
│ MEDIA PRIORIDAD      ┆ 312 │
│ ALTA PRIORIDAD       ┆ 86  │
│ OUT_OF_COHORT_REVIEW ┆ 5   │
└──────────────────────┴─────┘

Transacciones clasificadas como OUT_OF_COHORT_REVIEW:
shape: (5, 5)
┌────────────────┬────────┬──────────────────┬──────────────────────┬──────────────────────┐
│ transaction_id ┆ Amount ┆ Use Chip         ┆ priority_level       ┆ accion_recomendada   │
│ ---            ┆ ---    ┆ ---              ┆ ---                  ┆ ---                  │
│ i64            ┆ f64    ┆ str              ┆ str                  ┆ str                  │
╞════════════════╪════════╪══════════════════╪══════════════════════╪══════════════════════╡
│ 996            ┆ -75.5  ┆ Chip Transaction ┆ OUT_OF_COHORT_REVIEW ┆ Auditoría manual de  │


## 5. Exportación del Archivo de Predicciones
Exportamos `inference_results.csv` para la integración con sistemas de gestión de alertas.


In [7]:
export_cols = [
    "transaction_id",
    "User",
    "Card",
    "Year",
    "Month",
    "Day",
    "Time",
    "Amount",
    "Use Chip",
    "mcc_category",
    "fraud_probability",
    "risk_score",
    "prediction",
    "priority_level",
    "accion_recomendada",
]

df_export = df_results.select(export_cols).sort("risk_score", descending=True)
df_export.write_csv(INFERENCE_RESULTS_FILE)

print(f"Resultados de inferencia exportados exitosamente a: {INFERENCE_RESULTS_FILE}")
print(f"Dimensiones exportadas: {df_export.shape[0]} transacciones x {df_export.shape[1]} columnas")


Resultados de inferencia exportados exitosamente a: /home/glender/ProyectosUNi/iaupeu/data/predictions/inference_results.csv
Dimensiones exportadas: 1000 transacciones x 15 columnas
